##Q1


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

df = pd.read_csv(url)

df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [ ]:
df.shape

(10000, 11)

In [ ]:
df.isnull().sum()

,0
model_year,0
origin,0
fuel_type,0
drivetrain,0
num_doors,0
engine_displacement,0
num_cylinders,0
horsepower,877
vehicle_weight,0
acceleration,264


##Q2

In [ ]:
df["horsepower"].median()

254.0

##Q3

In [ ]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year"
]

target = "fuel_efficiency_mpg"

In [ ]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

idx = np.arange(n)

np.random.seed(42)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train+n_val]].copy()
df_test = df.iloc[idx[n_train+n_val:]].copy()

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

del df_train[target]
del df_val[target]
del df_test[target]

In [ ]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]


def rmse(y, y_pred):
    error = y - y_pred
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [ ]:
# ----- Fill missing horsepower with 0 -----

X_train_zero = df_train[features].fillna(0).values
X_val_zero = df_val[features].fillna(0).values

w0_zero, w_zero = train_linear_regression(X_train_zero, y_train)

y_pred_zero = w0_zero + X_val_zero.dot(w_zero)

rmse_zero = rmse(y_val, y_pred_zero)

print("RMSE with 0:", round(rmse_zero, 2))


# ----- Fill missing horsepower with TRAINING mean -----

mean_hp = df_train["horsepower"].mean()

X_train_mean = df_train[features].fillna({
    "horsepower": mean_hp
}).values

X_val_mean = df_val[features].fillna({
    "horsepower": mean_hp
}).values

w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)

y_pred_mean = w0_mean + X_val_mean.dot(w_mean)

rmse_mean = rmse(y_val, y_pred_mean)

print("Training horsepower mean:", mean_hp)
print("RMSE with mean:", round(rmse_mean, 2))

RMSE with 0: 2.21
Training horsepower mean: 254.46338337605272
RMSE with mean: 2.2


In [ ]:
# Calculate training means for all numerical columns
train_means = df_train[features].mean()

# Fill missing values with training means
X_train_mean = df_train[features].fillna(train_means).values
X_val_mean = df_val[features].fillna(train_means).values

w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)

y_pred_mean = w0_mean + X_val_mean.dot(w_mean)

rmse_mean = rmse(y_val, y_pred_mean)

print("RMSE with mean:", round(rmse_mean, 2))

RMSE with mean: 2.2


In [ ]:
print("RMSE with 0:", rmse_zero)
print("RMSE with mean:", rmse_mean)

RMSE with 0: 2.20529319475341
RMSE with mean: 2.2018174845688243


##Q4

In [ ]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]


train_means = df_train[features].mean()

X_train = df_train[features].fillna(train_means).values
X_val = df_val[features].fillna(train_means).values

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(
        X_train,
        y_train,
        r=r
    )

    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)

    print(r, score, "rounded:", round(score, 3))

0 2.2018174845688243 rounded: 2.202
0.01 2.2017619547150615 rounded: 2.202
0.1 2.2156413888659516 rounded: 2.216
1 2.3442243370702314 rounded: 2.344
5 2.4145466739025543 rounded: 2.415
10 2.4267527866241805 rounded: 2.427
100 2.438631664638164 rounded: 2.439


In [ ]:
# Use mean filling
train_means = df_train[features].mean()

X_train = df_train[features].fillna(train_means).values
X_val =df_val[features].fillna(train_means).values

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)

    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)

    print(r, round(score, 3))

0 2.202
0.01 2.202
0.1 2.216
1 2.344
5 2.415
10 2.427
100 2.439


##Q5

In [ ]:
scores = []

for seed in range(10):
    n = len(df)

    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    idx = np.arange(n)

    np.random.seed(seed)
    np.random.shuffle(idx)

    df_train_seed = df.iloc[idx[:n_train]].copy()
    df_val_seed = df.iloc[idx[n_train:n_train+n_val]].copy()

    y_train_seed = df_train_seed[target].values
    y_val_seed = df_val_seed[target].values

    del df_train_seed[target]
    del df_val_seed[target]

    train_means_seed = df_train_seed[numerical].mean()

    X_train_seed = df_train_seed[numerical].fillna(train_means_seed).values
    X_val_seed = df_val_seed[numerical].fillna(train_means_seed).values

    w0, w = train_linear_regression(X_train_seed, y_train_seed)

    y_pred = w0 + X_val_seed.dot(w)

    score = rmse(y_val_seed, y_pred)
    scores.append(score)

    print(seed, score)

std = np.std(scores)

print("RMSE standard deviation:", std)
print("Rounded:", round(std, 3))

0 2.2358078741149745
1 2.195618325482663
2 2.153812200808806
3 2.199500481084629
4 2.1920170819997153
5 2.2391247146208686
6 2.2640811215483927
7 2.177833142056664
8 2.2146681814268785
9 2.2017440400876307
RMSE standard deviation: 0.030487246763430093
Rounded: 0.03


In [ ]:
scores = []

for seed in range(10):
    n = len(df)

    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    idx = np.arange(n)

    np.random.seed(seed)
    np.random.shuffle(idx)

    df_train_seed = df.iloc[idx[:n_train]].copy()
    df_val_seed = df.iloc[idx[n_train:n_train+n_val]].copy()

    y_train_seed = df_train_seed[target].values
    y_val_seed = df_val_seed[target].values

    del df_train_seed[target]
    del df_val_seed[target]

    # Fill ALL missing numerical values with 0
    X_train_seed = df_train_seed[numerical].fillna(0).values
    X_val_seed = df_val_seed[numerical].fillna(0).values

    w0, w = train_linear_regression(X_train_seed, y_train_seed)

    y_pred = w0 + X_val_seed.dot(w)

    score = rmse(y_val_seed, y_pred)
    scores.append(score)

    print(seed, score)

std = np.std(scores)

print("RMSE standard deviation:", std)
print("Rounded:", round(std, 3))

0 2.2412812778301263
1 2.201882999653478
2 2.1589497421034385
3 2.2003788577324848
4 2.197077535231482
5 2.2456610733506355
6 2.2696410836928034
7 2.183920730360725
8 2.214278945075966
9 2.205285971730841
RMSE standard deviation: 0.030701494565491046
Rounded: 0.031


In [ ]:
scores = []

for seed in range(10):
    n = len(df)

    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    idx = np.arange(n)

    np.random.seed(seed)
    np.random.shuffle(idx)

    df_train_seed = df.iloc[idx[:n_train]].copy()
    df_val_seed = df.iloc[idx[n_train:n_train+n_val]].copy()

    y_train_seed = df_train_seed[target].values
    y_val_seed = df_val_seed[target].values

    X_train_seed = df_train_seed[features].fillna(0).values
    X_val_seed = df_val_seed[features].fillna(0).values

    w0, w = train_linear_regression(
        X_train_seed,
        y_train_seed
    )

    y_pred = w0 + X_val_seed.dot(w)

    score = rmse(y_val_seed, y_pred)
    scores.append(score)

    print(seed, score)

std = np.std(scores)

print("STD:", std)
print("Rounded:", round(std, 3))

0 2.239204143046126
1 2.2021128210838845
2 2.1634197787530947
3 2.2043588768539224
4 2.2018800943311554
5 2.2457851509085134
6 2.2697212079524043
7 2.190794599933443
8 2.216611201686444
9 2.2070365928178957
STD: 0.028781274078191175
Rounded: 0.029


##Q6

In [ ]:
# Q6

n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

idx = np.arange(n)

np.random.seed(9)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train+n_val]].copy()
df_test = df.iloc[idx[n_train+n_val:]].copy()

# Combine train + validation
df_train_full = pd.concat([df_train, df_val]).reset_index(drop=True)

y_train_full = df_train_full[target].values
y_test = df_test[target].values

X_train_full = df_train_full[features].fillna(0).values
X_test = df_test[features].fillna(0).values

# Train with regularization r = 0.001
w0, w = train_linear_regression_reg(
    X_train_full,
    y_train_full,
    r=0.001
)

y_pred = w0 + X_test.dot(w)

score = rmse(y_test, y_pred)

print("Test RMSE:", score)
print("Rounded:", round(score, 3))

Test RMSE: 2.235827747191731
Rounded: 2.236
